In [1]:
import pandas as pd
import json
import os
import glob
from collections import Counter

data_folder = r"C:\Research\EndpointXAI\data"
all_files = glob.glob(data_folder + "/**/*.json", recursive=True)
print(f"Found {len(all_files)} files")

Found 31 files


In [2]:
# Scan all files to find every possible field
all_keys = Counter()

for file in all_files:
    try:
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    record = json.loads(line)
                    for key in record.keys():
                        all_keys[key] += 1
    except Exception as e:
        pass

# Show top 30 most common fields
print("Top 30 fields across ALL files:")
for key, count in all_keys.most_common(30):
    print(f"  {key}: {count} records")

Top 30 fields across ALL files:
  @timestamp: 97608 records
  host: 97608 records
  port: 80218 records
  Severity: 80218 records
  EventTime: 80218 records
  SourceName: 80218 records
  Channel: 80218 records
  EventReceivedTime: 80218 records
  EventID: 80218 records
  SeverityValue: 80218 records
  Keywords: 80218 records
  EventType: 80218 records
  SourceModuleName: 80218 records
  SourceModuleType: 80218 records
  RecordNumber: 80218 records
  ThreadID: 80218 records
  Task: 80218 records
  Hostname: 80218 records
  @version: 80218 records
  OpcodeValue: 79192 records
  ProviderGuid: 79192 records
  Version: 79192 records
  Message: 78991 records
  Opcode: 78585 records
  Category: 78420 records
  ExecutionProcessID: 77979 records
  tags: 77979 records
  ProcessId: 74205 records
  AccountName: 70057 records
  AccountType: 70057 records


In [3]:
# Save full field list to a text file so we can see everything
with open(r"C:\Research\EndpointXAI\data\all_fields.txt", "w") as f:
    f.write("All fields across all files:\n\n")
    for key, count in all_keys.most_common():
        f.write(f"{key}: {count} records\n")

print(f"Total unique fields: {len(all_keys)}")
print(f"\nFull list saved to all_fields.txt")
print(f"\nFields with 50,000+ records:")
for key, count in all_keys.most_common():
    if count >= 50000:
        print(f"  {key}: {count}")

Total unique fields: 238

Full list saved to all_fields.txt

Fields with 50,000+ records:
  @timestamp: 97608
  host: 97608
  port: 80218
  Severity: 80218
  EventTime: 80218
  SourceName: 80218
  Channel: 80218
  EventReceivedTime: 80218
  EventID: 80218
  SeverityValue: 80218
  Keywords: 80218
  EventType: 80218
  SourceModuleName: 80218
  SourceModuleType: 80218
  RecordNumber: 80218
  ThreadID: 80218
  Task: 80218
  Hostname: 80218
  @version: 80218
  OpcodeValue: 79192
  ProviderGuid: 79192
  Version: 79192
  Message: 78991
  Opcode: 78585
  Category: 78420
  ExecutionProcessID: 77979
  tags: 77979
  ProcessId: 74205
  AccountName: 70057
  AccountType: 70057
  Domain: 70057
  UserID: 70057
  UtcTime: 67792
  RuleName: 66162
  Image: 65504
  ProcessGuid: 65504
  EventTypeOrignal: 57336
  TargetObject: 57177


In [5]:
# Rebuild dataset - ONLY from real attack data folders
all_records = []

# Filter to only include files in datasets/atomic/windows/
attack_files = [f for f in all_files if 'datasets' in f.replace('\\','/') 
                and 'atomic' in f.replace('\\','/') 
                and 'windows' in f.replace('\\','/')]

print(f"Filtered to {len(attack_files)} actual attack data files")

useful_fields = [
    'EventID', 'Task', 'Channel', 'Opcode', 'OpcodeValue',
    'SourceName', 'SourceModuleName', 'EventType',
    'Severity', 'SeverityValue', 'Hostname',
    'ProcessId', 'ThreadID', 'AccountName', 'AccountType',
    'Message', 'Keywords'
]

skipped_count = 0
for file in attack_files:
    try:
        parts = file.replace('\\', '/').split('/')
        try:
            windows_idx = parts.index('windows')
            folder_name = parts[windows_idx + 1]
        except:
            folder_name = 'unknown'
        
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        record = json.loads(line)
                        clean = {field: record.get(field, None) for field in useful_fields}
                        clean['attack_label'] = folder_name
                        all_records.append(clean)
                    except:
                        continue
    except Exception as e:
        skipped_count += 1

df = pd.DataFrame(all_records)
print(f"\nTotal events: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Skipped files: {skipped_count}")
print(f"\nAttack label distribution:")
print(df['attack_label'].value_counts())

Filtered to 17 actual attack data files

Total events: 97611
Total columns: 18
Skipped files: 0

Attack label distribution:
attack_label
lateral_movement    77982
defense_evasion     17390
persistence          2239
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import LabelEncoder

# Fill missing values
df = df.fillna('unknown')

# Convert all text columns to numbers
encoders = {}
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

print("All columns converted to numbers ✅")
print(f"\nFinal shape: {df.shape}")
print(df.head())

All columns converted to numbers ✅

Final shape: (97611, 18)
   EventID  Task  Channel   Opcode  OpcodeValue SourceName SourceModuleName  \
0       64    47  unknown  unknown            7    unknown          unknown   
1       64    47  unknown  unknown            7    unknown          unknown   
2       64    47  unknown  unknown            7    unknown          unknown   
3       64    47  unknown  unknown            7    unknown          unknown   
4       64    47  unknown  unknown            7    unknown          unknown   

  EventType Severity  SeverityValue Hostname ProcessId  ThreadID AccountName  \
0   unknown  unknown              4  unknown   unknown       116     unknown   
1   unknown  unknown              4  unknown   unknown       116     unknown   
2   unknown  unknown              4  unknown   unknown       116     unknown   
3   unknown  unknown              4  unknown   unknown       116     unknown   
4   unknown  unknown              4  unknown   unknown       116

In [10]:
from sklearn.model_selection import train_test_split

# Save the new rich dataset
df.to_csv(r"C:\Research\EndpointXAI\data\mordor_rich_features.csv", index=False)
print("Saved rich dataset ✅")

# Split for training
X = df.drop('attack_label', axis=1)
y = df['attack_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Overwrite the old splits with rich features
X_train.to_csv(r"C:\Research\EndpointXAI\data\X_train.csv", index=False)
X_test.to_csv(r"C:\Research\EndpointXAI\data\X_test.csv", index=False)
y_train.to_csv(r"C:\Research\EndpointXAI\data\y_train.csv", index=False)
y_test.to_csv(r"C:\Research\EndpointXAI\data\y_test.csv", index=False)

print(f"\nNew train shape: {X_train.shape}")
print(f"New test shape: {X_test.shape}")
print("\n Ready to retrain models!")

Saved rich dataset ✅

New train shape: (78088, 17)
New test shape: (19523, 17)

 Ready to retrain models!
